# 01 Symbol Backtest

Notebook này dùng để backtest một symbol thật kỹ, nhưng workflow chính giờ nằm trong **bảng điều khiển tương tác** ở Cell 4.

Bạn chỉ cần:
1. Chạy Cell 1 → Cell 4.
2. Chọn symbol, account mode, date range, max bars.
3. Bấm **Run selected symbol** để xem báo cáo đầy đủ.
4. Bấm **Compare selected symbols** để so sánh nhiều symbol theo cùng một khoảng thời gian và config hiện tại.

Các cell sau Cell 4 chỉ là phần tùy chọn: Monte Carlo và export kết quả.

In [ ]:
# Cell 1 - Bootstrap đường dẫn import an toàn
#
# Notebook có thể được mở từ repo root, từ thư mục strategies/combo,
# hoặc từ một working directory khác trong VS Code/Jupyter. Vì vậy ta không
# dùng `config.py` làm marker root vì trong strategies/combo cũng có config.py.

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Không tìm thấy repo root chứa {marker!r} và core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)

In [ ]:
# Cell 2 - Import thư viện, runner và helper hiển thị
#
# File `notebook_utils.py` chịu trách nhiệm render dashboard, bảng so sánh và
# widget. Notebook này vì vậy gọn hơn: phần logic hiển thị không còn nằm rải rác
# trong nhiều cell.

from IPython.display import display

from shared.monte_carlo import plot_monte_carlo, run_monte_carlo
from strategies.combo.config import SYMBOLS, summary as strategy_summary
from strategies.combo.notebook_utils import (
    build_symbol_backtest_widget,
    configure_notebook,
    export_result_bundle,
    show_run_config,
)
from strategies.combo.symbol.backtest import run_symbol_backtest

configure_notebook()
print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))

In [ ]:
# Cell 3 - Cấu hình mặc định cho bảng điều khiển
#
# Đây là giá trị khởi tạo cho dropdown ở Cell 4. Khi bạn bấm Run trong widget,
# RUN_CONFIG sẽ được cập nhật lại theo giá trị hiện đang chọn.

RUN_CONFIG = {
    'symbol': 'US30',
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'date_from': '2023-01-01',
    'date_to': None,
    'max_bars': 50_000,
    'indicator_overrides': {},
    'symbol_overrides': {},
    'export_report': False,
}

show_run_config('Cấu hình mặc định', RUN_CONFIG)

In [ ]:
# Cell 4 - Bảng điều khiển tương tác
#
# Single-symbol backtest:
# - Chọn symbol/account/date/max_bars.
# - Bấm Run selected symbol.
# - Kết quả sẽ tự hiển thị: header, KPI cards, KPI chi tiết, monthly PnL,
#   equity/drawdown/trade PnL và trade summary.
#
# Compare symbols:
# - Chọn nhiều symbol trong khung Compare.
# - Bấm Compare selected symbols.
# - Notebook chạy từng symbol theo cùng config hiện tại và hiển thị bảng so sánh.

symbol_dashboard = build_symbol_backtest_widget(
    symbols=SYMBOLS,
    run_symbol_backtest=run_symbol_backtest,
    default_config=RUN_CONFIG,
    global_ns=globals(),
)
display(symbol_dashboard)

In [ ]:
# Cell 5 - Monte Carlo robustness cho kết quả vừa chạy
#
# Cell này dùng biến `result` do Cell 4 tạo ra sau khi bấm Run selected symbol.
# Nếu chưa bấm Run, cell sẽ nhắc bạn chạy Cell 4 trước.

if 'result' not in globals():
    print('Chưa có result. Hãy bấm Run selected symbol ở Cell 4 trước.')
elif not result.trades:
    print('Bỏ qua Monte Carlo vì result hiện tại không có trade.')
else:
    trade_pnls = [float(t['pnl_usd']) for t in result.trades]
    mc = run_monte_carlo(
        trade_pnls,
        n_iter=500,
        dd_threshold=0.20,
        initial_balance=RUN_CONFIG['initial_balance'],
    )
    print('Monte Carlo P(max DD > 20%) =', round(mc['prob_exceed_dd'] * 100, 2), '%')
    print('Sharpe CI 95% =', (round(mc['sharpe_ci_low'], 2), round(mc['sharpe_ci_high'], 2)))
    plot_monte_carlo(mc)

In [ ]:
# Cell 6 - Export kết quả nếu cần
#
# Cell này cũng dùng biến `result` do Cell 4 tạo ra.
# Mặc định export tắt để tránh tạo file ngoài ý muốn.

EXPORT_REPORT = False

if 'result' not in globals():
    print('Chưa có result. Hãy bấm Run selected symbol ở Cell 4 trước.')
elif EXPORT_REPORT:
    export_result_bundle(
        f"{result.symbol}_{result.account_mode}_symbol_backtest",
        metrics=result.metrics,
        trades=result.trades,
        equity=result.equity,
    )
else:
    print('Export đang tắt. Đổi EXPORT_REPORT = True nếu muốn lưu CSV.')